# 07 — Compound optimization (planner inside MetaOrchestrator)

**Same machinery, more compound surface.**

Notebook 06 evolved the prompt of one isolated agent. This notebook evolves the *planner* inside `MetaOrchestrator` and watches the entire orchestration improve downstream — the planner's task decompositions get cleaner, which means each spawned sub-agent gets a sharper subtask, which means the end-to-end success rate climbs.

Score function operates one layer up: we compare the planner's emitted `TaskDecomposition` against an expected shape. The downstream effect is then visible in the optional final cell that runs full orchestrations before vs after.

**Prerequisites:** same as notebook 06. Cost: ~$2–$3.

## 1. Load config + define the planner

In [1]:
from typing import Any

from pydantic import BaseModel

from orqest import load_config
from orqest.agents import BaseAgent, GlobalState
from orqest.autonomy.meta import TaskDecomposition

config = load_config()
print(f"Model: {config.llm_model}")

INITIAL_PLANNER_PROMPT = (
    "You decompose a high-level goal into 2-4 concrete subtasks. "
    "Each subtask needs a short snake_case name, a one-sentence description, "
    "and requires_agent=True. Keep the list minimal."
)

class PlannerAgent(BaseAgent[GlobalState, TaskDecomposition]):
    async def _run_implementation(self, state: GlobalState, **kwargs: Any) -> TaskDecomposition:
        latest = state.get_latest_message("user") or ""
        result = await self.call_model(latest, state)
        return result.output

Model: openai:gpt-5.2


## 2. The 15-example gold set

Three buckets:
- **Single-step (5)** — goal needs *one* specialist; planner should not over-decompose.
- **Multi-step (5)** — goal needs 3+ subtasks; planner should produce a coherent ordering.
- **Ambiguous (5)** — goal underspecified; planner should pick a conservative minimal decomposition rather than fabricating scope.

We score by comparing subtask *count* against an expected range plus a name-quality heuristic — not exact-match (LLMs vary on naming) but tight enough to track structural improvements.

In [2]:
from orqest.optimization import GoldExample

class PlannerExpected(BaseModel):
    min_subtasks: int
    max_subtasks: int
    must_mention: list[str] = []   # snake_case keywords any subtask name should contain at least one of

def _ex(goal: str, lo: int, hi: int, mention: list[str], bucket: str) -> GoldExample:
    return GoldExample[str, PlannerExpected](
        input=goal,
        expected=PlannerExpected(min_subtasks=lo, max_subtasks=hi, must_mention=mention),
        id=f"{bucket}-{hash(goal) % 10000}",
    )

EXAMPLES = [
    # Single-step (1-2 subtasks)
    _ex("Translate this sentence to French: 'The cat sat on the mat.'", 1, 2, ["translat"], "single"),
    _ex("What is 13 * 47?", 1, 2, ["calc", "comput", "answer"], "single"),
    _ex("Define the word 'antidisestablishmentarianism'.", 1, 2, ["defin", "explain"], "single"),
    _ex("Sort this list alphabetically: zebra, apple, mango, banana.", 1, 2, ["sort"], "single"),
    _ex("What's the capital of France?", 1, 2, ["answer", "lookup", "capital"], "single"),
    # Multi-step (3-4 subtasks)
    _ex("Research the latest electric vehicle market, summarize the top 3 manufacturers, and recommend one for a family of 5.", 3, 4, ["research", "summar", "recommend"], "multi"),
    _ex("Build a meal plan: gather dietary requirements, propose a 7-day menu, then list the shopping items.", 3, 4, ["requir", "menu", "shop"], "multi"),
    _ex("Plan a 3-day trip to Tokyo: research attractions, build a daily itinerary, and estimate the total budget.", 3, 4, ["research", "itinerary", "budget"], "multi"),
    _ex("Diagnose a slow API: gather logs, identify the bottleneck, propose a fix, and validate against test cases.", 3, 4, ["log", "bottleneck", "fix", "valid"], "multi"),
    _ex("Design a CLI for converting CSV to JSON: name subcommands, list global flags, write a usage example.", 3, 4, ["subcommand", "flag", "example"], "multi"),
    # Ambiguous (2-3 subtasks — conservative)
    _ex("Help me with my project.", 2, 3, ["clarif", "requir", "scope"], "ambiguous"),
    _ex("Make this better.", 2, 3, ["clarif", "context", "requir"], "ambiguous"),
    _ex("I need some advice.", 2, 3, ["clarif", "context", "topic"], "ambiguous"),
    _ex("Tell me about it.", 2, 3, ["clarif", "context", "subject"], "ambiguous"),
    _ex("Fix the issue.", 2, 3, ["clarif", "identif", "context"], "ambiguous"),
]
print(f"Loaded {len(EXAMPLES)} planner gold examples.")

Loaded 15 planner gold examples.


## 3. Wrap the planner in an Evaluator

Score function checks subtask count is in `[min, max]` and that at least one expected keyword appears across subtask names.

In [3]:
from orqest.optimization import Evaluator

def make_planner(decoded: dict[str, Any]) -> PlannerAgent:
    return PlannerAgent(
        agent_name="planner",
        system_prompt=decoded["planner.system_prompt"],
        output_type=TaskDecomposition,
        model=config.llm_model,
        api_key=config.llm_api_key,
    )

def score(output: TaskDecomposition, ex: GoldExample[str, PlannerExpected]) -> float:
    expected = ex.expected
    n = len(output.subtasks)
    count_ok = expected.min_subtasks <= n <= expected.max_subtasks

    name_blob = " ".join(s.name.lower() for s in output.subtasks)
    mention_ok = (not expected.must_mention) or any(
        keyword in name_blob for keyword in expected.must_mention
    )

    if count_ok and mention_ok:
        return 1.0
    if count_ok or mention_ok:
        return 0.5
    return 0.0

evaluator = Evaluator(agent_factory=make_planner, score_fn=score)
print("Evaluator ready.")

Evaluator ready.


## 4. Baseline — what does the unoptimized planner do?

In [4]:
import statistics
from collections import defaultdict

async def measure(decoded: dict[str, Any]) -> dict[str, float]:
    bundles = await evaluator.evaluate_batch(decoded, EXAMPLES)
    by_bucket: dict[str, list[float]] = defaultdict(list)
    for ex, b in zip(EXAMPLES, bundles, strict=True):
        bucket = ex.id.split("-")[0]
        by_bucket[bucket].append(b.accuracy)
    return {bucket: statistics.mean(scores) for bucket, scores in by_bucket.items()} | {
        "overall": statistics.mean(b.accuracy for b in bundles)
    }

baseline = await measure({"planner.system_prompt": INITIAL_PLANNER_PROMPT})
print("Baseline (planner structural quality):")
for k, v in baseline.items():
    print(f"  {k:14s} {v:.3f}")

Baseline (planner structural quality):
  single         0.400
  multi          1.000
  ambiguous      0.900
  overall        0.767


## 5. Define the genome — one prompt slot, layered constraints

In [5]:
from orqest.optimization import Genome, PromptGene

genome = Genome(genes=[
    PromptGene(
        name="planner.system_prompt",
        initial=INITIAL_PLANNER_PROMPT,
        constraints=(
            "Match decomposition depth to goal complexity: 1-2 subtasks for trivial goals, "
            "3-4 for genuinely multi-step goals, 2-3 for ambiguous goals (where the first "
            "subtask should typically be clarifying scope rather than fabricating it)."
        ),
    ),
])

## 6. Run the optimizer

In [6]:
from orqest.optimization import OptimizationConfig, OptimizationRunner

opt_config = OptimizationConfig(
    max_metric_calls=80,
    reflection_model=config.llm_model,
    # No task_model — the agent_factory above wires the model into the planner itself.
    # GEPA's optimize() asserts task_lm is None when an adapter is provided.
    minibatch_size=3,
    valset_fraction=0.4,
    seed=42,
)

# api_key bridges to litellm's expected env var for the reflection LLM
runner = OptimizationRunner(
    opt_config,
    genome=genome,
    evaluator=evaluator,
    api_key=config.llm_api_key,
)
result = await runner.optimize(EXAMPLES)
print(f"Best aggregate score: {result.best_score:.3f}")
print(f"Pareto frontier size: {len(result.pareto_candidates)}")

Iteration 0: Base program full valset score: 0.926317559593396 over 6 / 6 examples
Iteration 1: Selected program 0 score: 0.926317559593396


Iteration 1: Proposed new text for planner.system_prompt: You are a task-decomposition assistant.

Given a single user “goal” (one short request), output ONLY a minimal JSON array of 2–4 subtasks (no extra text, no headings, no code fences).

Each subtask object must have exactly:
- name: short snake_case identifier (2–5 words, no punctuation)
- description: exactly one sentence describing the concrete action
- requires_agent: true (literal boolean)

Decomposition rules (match depth to goal complexity):
- Trivial, single-step, unambiguous goals (e.g., “sort this list”, “define a word”): use 1–2 subtasks (prefer 1 when truly atomic).
- Ambiguous goals (e.g., “Help me with my project”): use 2–3 subtasks, and the FIRST subtask must be to clarify scope/requirements (ask what, constraints, desired output), not to invent specifics.
- Genuinely multi-step goals: use 3–4 subtasks.

Keep subtasks concrete and sequential (each enables the next), avoid redundant granularity, and do not fabricate 

Iteration 1: New subsample score 2.5250026491389144 is better than old score 2.3096762337203836. Continue to full eval and add to candidate pool.


Iteration 1: Valset score for new program: 0.7828810200732551 (coverage 6 / 6)
Iteration 1: Val aggregate for new program: 0.7828810200732551
Iteration 1: Individual valset scores for new program: {2: 0.6518461826996644, 3: 0.9508590889797779, 0: 0.9017835869197733, 1: 0.912033348220284, 4: 0.8749846423801501, 5: 0.40577927123988045}
Iteration 1: Objective aggregate scores for new program: {'accuracy': 0.9166666666666666, 'cost_usd': 0.0, 'latency_ms': 6689.28232967058}
Iteration 1: New valset pareto front scores: {0: 0.9183641041599913, 1: 0.9213550742599181, 2: 0.9044824173202506, 3: 0.9508590889797779, 4: 0.9442665028799092, 5: 0.9324069959000917}
Iteration 1: Objective pareto front scores: {'accuracy': 1.0, 'cost_usd': 0.0, 'latency_ms': 6689.28232967058}
Iteration 1: Valset pareto front aggregate score: 0.9286223639166565
Iteration 1: Updated valset pareto front programs: {0: {0}, 1: {0}, 2: {0}, 3: {1}, 4: {0}, 5: {0}}
Iteration 1: Updated objective pareto front programs: {'accur

Iteration 2: Proposed new text for planner.system_prompt: You are a task-decomposition assistant.

Input: a single user “goal” as one short request.

Output: ONLY a valid, minimal JSON array of 2–4 sequential subtasks (no prose, no headings, no markdown/code fences, no extra keys).

Hard requirements for each subtask object:
- name: a short snake_case identifier of 2–5 words; use only letters/numbers/underscores; no punctuation/hyphens.
- description: exactly ONE grammatical sentence describing a concrete action to take (not a fragment, not multiple sentences).
- requires_agent: true (literal boolean, not a string).

Decomposition rules:
- If the goal is ambiguous/vague (e.g., “Make this better.”, “Help me with my project.”), the FIRST subtask MUST be to ask clarifying questions about scope/requirements (what the item is, success criteria, constraints, desired output format, audience, tone, length, etc.) and MUST NOT invent specifics.
- If the goal is trivial and truly atomic, still re

...

Iteration 7: New subsample score 2.754299490819103 is better than old score 1.7173174342396669. Continue to full eval and add to candidate pool.


Iteration 7: Valset score for new program: 0.915364568666361 (coverage 6 / 6)
Iteration 7: Val aggregate for new program: 0.915364568666361
Iteration 7: Individual valset scores for new program: {0: 0.9446840500796679, 5: 0.8976319361996139, 1: 0.8690879095596028, 2: 0.8995476897194749, 3: 0.9406222094397526, 4: 0.9406136170000536}
Iteration 7: Objective aggregate scores for new program: {'accuracy': 1.0, 'cost_usd': 0.0, 'latency_ms': 4231.771566681952}
Iteration 7: New valset pareto front scores: {0: 0.9569423183600884, 1: 0.9336820579401683, 2: 0.9488050535001094, 3: 0.9508590889797779, 4: 0.957842073480133, 5: 0.9324069959000917}
Iteration 7: Objective pareto front scores: {'accuracy': 1.0, 'cost_usd': 0.0, 'latency_ms': 6689.28232967058}
Iteration 7: Valset pareto front aggregate score: 0.9467562646933948
Iteration 7: Updated valset pareto front programs: {0: {3}, 1: {3}, 2: {4}, 3: {1}, 4: {3}, 5: {0}}
Iteration 7: Updated objective pareto front programs: {'accuracy': {0, 2, 4, 6

Iteration 8: Proposed new text for planner.system_prompt: Decompose the user’s high-level goal into a minimal list of concrete subtasks.

Output format:
- Return ONLY a JSON array (no prose, no extra keys, no code fences).
- Each array item is an object with exactly these keys:
  - name: short snake_case identifier
  - description: one sentence describing the subtask
  - requires_agent: boolean, always true

Decomposition rules:
- Use 1–2 subtasks for trivial, single-hop goals (e.g., definitions, simple facts, simple sorting).
- Use 3–4 subtasks only for genuinely multi-step goals.
- Use 2–3 subtasks for ambiguous goals; the first subtask should clarify scope/requirements rather than inventing details.
- Keep the list minimal: do not add “research/verify” steps unless the goal truly requires it.
- Make subtasks concrete and action-oriented; avoid redundant or overlapping steps.


Iteration 8: New subsample score 2.766526012820541 is better than old score 1.8179392307595117. Continue to full eval and add to candidate pool.


Iteration 8: Valset score for new program: 0.922537339879976 (coverage 6 / 6)
Iteration 8: Val aggregate for new program: 0.922537339879976
Iteration 8: Individual valset scores for new program: {2: 0.9466932054201607, 3: 0.8526080972002819, 4: 0.9672247102000984, 0: 0.928263230919838, 1: 0.9017831953795394, 5: 0.9386516001599375}
Iteration 8: Objective aggregate scores for new program: {'accuracy': 1.0, 'cost_usd': 0.0, 'latency_ms': 3873.133006001202}
Iteration 8: New valset pareto front scores: {0: 0.9569423183600884, 1: 0.9336820579401683, 2: 0.9488050535001094, 3: 0.9508590889797779, 4: 0.9672247102000984, 5: 0.9386516001599375}
Iteration 8: Objective pareto front scores: {'accuracy': 1.0, 'cost_usd': 0.0, 'latency_ms': 6689.28232967058}
Iteration 8: Valset pareto front aggregate score: 0.9493608048566967
Iteration 8: Updated valset pareto front programs: {0: {3}, 1: {3}, 2: {4}, 3: {1}, 4: {7}, 5: {7}}
Iteration 8: Updated objective pareto front programs: {'accuracy': {0, 2, 4, 6

## 7. Diff + per-bucket lift

In [7]:
from orqest.optimization import apply_result

baseline_planner = make_planner({"planner.system_prompt": INITIAL_PLANNER_PROMPT})
for d in apply_result(result, target=baseline_planner):
    if d.changed:
        print(d.unified)

evolved = await measure(result.best_decoded)
print("\nPer-bucket structural quality:")
print(f"  {'bucket':<12s} {'before':>10s} {'after':>10s} {'delta':>10s}")
for bucket in ("single", "multi", "ambiguous", "overall"):
    b, a = baseline.get(bucket, 0.0), evolved.get(bucket, 0.0)
    arrow = "↑" if a > b else ("↓" if a < b else "=")
    print(f"  {bucket:<12s} {b:>10.3f} {a:>10.3f}     {a - b:+.3f} {arrow}")


Per-bucket structural quality:
  bucket           before      after      delta
  single            0.400      0.700     +0.300 ↑
  multi             1.000      1.000     +0.000 =
  ambiguous         0.900      1.000     +0.100 ↑
  overall           0.767      0.900     +0.133 ↑


## 8. Downstream effect — does the orchestration actually get better?

The planner score is upstream of orchestration success. We've improved the structural metric; let's see whether the cleaner decompositions translate to a better end-to-end run on a held-out goal.

Honest framing: planner improvement directly drives a structural metric. The downstream effect is noisier — many other factors (sub-agent prompts, tool quality, model variance) shape the final result. Don't expect dramatic single-goal lifts; do expect cleaner subtask graphs and fewer retries.

In [8]:
from orqest.autonomy import AgentFactory, MetaOrchestrator, ToolRegistry

registry = ToolRegistry()
factory = AgentFactory(registry, default_model=config.llm_model, api_key=config.llm_api_key)

HELD_OUT_GOAL = (
    "Plan a small science fair project for an 8th-grader: pick a topic, "
    "design the experiment, list the materials, write a one-paragraph abstract."
)

async def run_orchestration(planner_prompt: str) -> dict[str, Any]:
    planner = make_planner({"planner.system_prompt": planner_prompt})
    meta = MetaOrchestrator(planner, factory, registry, max_subtasks=5)
    res = await meta.solve(HELD_OUT_GOAL)
    return {
        "success": res.success,
        "n_subtasks": len(res.subtask_results),
        "successful_subtasks": sum(1 for r in res.subtask_results if r.success),
        "duration_ms": res.total_duration_ms,
    }

before = await run_orchestration(INITIAL_PLANNER_PROMPT)
after = await run_orchestration(result.best_decoded["planner.system_prompt"])
print(f"BEFORE: {before}")
print(f"AFTER:  {after}")

BEFORE: {'success': True, 'n_subtasks': 4, 'successful_subtasks': 4, 'duration_ms': 32829.099323018454}
AFTER:  {'success': True, 'n_subtasks': 4, 'successful_subtasks': 4, 'duration_ms': 23345.059368992224}


## What's next

- **Compose with healing** — feed `metacognition.confidence` into `MetricBundle.confidence` and let the optimizer evolve a planner that's both more accurate AND more confident in its decompositions. The plumbing is already there; just pass `confidence_protocol=StructuredOutputProtocol()` to the `Evaluator`.
- **W3 — topology IR.** Once `TopologySpec` ships, the optimizer can mutate not just *what* the planner says but *how the orchestrator wires* the spawned agents (parallel vs sequential, with vs without `RefinementLoop` wrapping). Same `MetricBundle` Pareto contract, much larger search space.
- See `docs/concepts/optimization.md` for the full architecture.